In [14]:
from __future__ import annotations
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import os

### Читаем все данные

In [15]:
X_screen = pd.read_csv("datasets/all_features_dataset_screen_vasiliy.csv", index_col=0).replace([np.inf, -np.inf], np.nan)
X_answer = pd.read_csv("datasets/all_features_dataset_answer_vasiliy.csv", index_col=0).replace([np.inf, -np.inf], np.nan)

participants_meta = pd.read_csv('datasets/CB_EEG_RSF.csv', decimal=',')[[
    'ID', 
    'IAT_results', 
    'IAT_result_word', 
    'IAT_results2', 
    'Gender', 
    'Region', 
    'Age',
    'Employed_media', 
    'Employed_medicine',
    'Interest_medicine',
    'EXP_general'
    ]]

text_features = pd.read_csv('datasets/Stimuli.csv', index_col=0)[
    ['Row_number',
     'Dlina']
].rename(columns={'Row_number':'TextID', 'Dlina':'text_length'})

stimul_features = pd.read_csv('datasets/stimulus_features.csv', index_col=0)
stimul_features['Index'] = stimul_features.index

stimul_features = stimul_features.merge(text_features, on='TextID')
stimul_features = stimul_features.drop(columns=['TextID'])
stimul_features = stimul_features.set_index('Index')
stimul_features.index.name=None
stimul_features = stimul_features.rename({'Performance Rate1': 'answer', 'Valence': 'valence', 'Veracity': 'veracity'}, axis=1)
stimul_features = stimul_features.drop(columns=['Main_ Answer'])
stimul_features['valence'] = stimul_features['valence'].map({0:2, 1:0, 2:1}) # 0-negative, 1-positive, 2-neutral
stimul_features['answer'] = stimul_features['answer'] - 1
stimul_features

,valence,veracity,answer,text_length
11101_105,0,2,2,88
11101_109,1,2,2,73
11101_60,1,1,7,91
11101_64,0,1,2,121
11101_73,0,2,1,83
...,...,...,...,...
22137_108,1,2,0,82
22137_16,2,1,6,57
22137_101,1,2,0,50
22137_30,2,1,6,56


In [5]:
participants_meta['EXP_general'].unique()

array([1.58333333, 2.        , 2.83333333, 1.95833333, 1.33333333,
       4.33333333, 2.5       , 2.41666667, 2.79166667, 1.83333333,
       4.25      , 2.66666667, 1.08333333, 3.41666667, 2.33333333,
       3.33333333, 2.16666667, 2.08333333, 3.16666667, 2.25      ,
       2.75      , 2.58333333, 3.25      , 1.66666667, 2.91666667,
       1.5       , 1.25      , 2.70833333, 5.04166667, 3.58333333,
       2.20833333, 1.45833333, 5.41666667, 3.45833333, 4.        ,
       3.125     , 4.41666667,        nan])

In [16]:
TEST_SIZE = 0.2
RANDOM_STATE = 1717
people_to_remove = [21001, 21007, 21010, 22009, 22017, 22019]

X_screen = X_screen.loc[[int(i.split('_')[0]) not in people_to_remove for i in X_screen.index]]
X_answer = X_answer.loc[[int(i.split('_')[0]) not in people_to_remove for i in X_answer.index]]

participants_in_X_all = [int(i.split('_')[0]) for i in X_screen.index] + [int(i.split('_')[0]) for i in X_answer.index]

participants_meta = participants_meta[participants_meta['ID'].isin(participants_in_X_all)]
participants_meta = participants_meta[~participants_meta['ID'].isin(people_to_remove)]

In [17]:
def _build_stimulus_norm(X: pd.DataFrame, train_participants: list[int]):
    # парсим participant / stim_id из индекса
    parts = X.index.to_series().str.split('_', expand=True)
    part_id = parts[0].astype(int)
    stim_id = parts[1].astype(int)

    # берём только строки train-участников
    mask_train_parts = part_id.isin(train_participants)
    X_train_parts = X.loc[mask_train_parts]

    stim_id_train = stim_id.loc[mask_train_parts]

    # считаем per-stimulus mean/std по ИСКЛЮЧИТЕЛЬНО колонкам X
    tmp = X_train_parts.copy()
    tmp["__stim__"] = stim_id_train.values
    by_stim_mean = tmp.groupby("__stim__")[X.columns].mean()
    by_stim_std  = tmp.groupby("__stim__")[X.columns].std(ddof=0)

    # overall по train-участникам
    overall_mean = X_train_parts.mean(axis=0)
    overall_std  = X_train_parts.std(axis=0, ddof=0)

    # защита от нулевых std: заменяем на overall_std (а их нули — на 1.0)
    by_stim_std = by_stim_std.replace(0, np.nan)
    overall_std = overall_std.replace(0, np.nan)

    return by_stim_mean, by_stim_std, overall_mean, overall_std


def _apply_stimulus_norm(subX: pd.DataFrame,
                         by_stim_mean: pd.DataFrame,
                         by_stim_std: pd.DataFrame,
                         overall_mean: pd.Series,
                         overall_std: pd.Series) -> pd.DataFrame:
    # парсим стимулы
    stim_ids = subX.index.to_series().str.split('_', expand=True)[1].astype(int)

    # жёстко выравниваем колонки
    cols = subX.columns
    by_stim_mean = by_stim_mean.reindex(columns=cols)
    by_stim_std  = by_stim_std.reindex(columns=cols)
    overall_mean = overall_mean.reindex(cols)
    overall_std  = overall_std.reindex(cols)

    # собираем построчные матрицы mu/std по stim_ids
    mu_df = by_stim_mean.reindex(stim_ids).copy()
    sg_df = by_stim_std.reindex(stim_ids).copy()
    # индексы под вычитание
    mu_df.index = subX.index
    sg_df.index = subX.index

    # заполняем пропуски overall-статистиками
    mu_df = mu_df.fillna(overall_mean)
    sg_df = sg_df.fillna(overall_std)

    # окончательная защита от нулевых/NaN std
    sg_df = sg_df.replace(0, np.nan).fillna(1.0)

    Z = (subX - mu_df) / sg_df
    Z = Z.replace([np.inf, -np.inf], 0.0).fillna(0.0)
    return Z

### Делим дынные ПО УЧАСТНИКАМ

In [18]:
train_participants, test_participants = train_test_split(participants_meta['ID'].tolist(), 
                                                         test_size=TEST_SIZE, 
                                                         stratify = participants_meta['IAT_results2'].tolist(),
                                                         random_state=RANDOM_STATE)


def split_cb_and_save(X, X_name, name):
    # name is 'binary' or 'multiclass' (used only for targets)
    base = f'datasets_splitted_{X_name}_cognitive_bias'
    os.makedirs(base, exist_ok=True)

    participants_in_X = [int(i.split('_')[0]) for i in X.index]

    y_reg = pd.Series([participants_meta[participants_meta['ID']==i]['IAT_results'].item()
                       for i in participants_in_X], name='IAT_score', index=X.index)
    y_binary = pd.Series([participants_meta[participants_meta['ID']==i]['IAT_result_word'].item()
                          for i in participants_in_X], name='IAT_binary_label', index=X.index).map({'Negative':0, 'Positive':1})
    y_multiclass = pd.Series([participants_meta[participants_meta['ID']==i]['IAT_results2'].item()
                              for i in participants_in_X], name='IAT_multi_label', index=X.index).map({'Negative':0,'Positive':1,'Neutral':2})

    stimuli_features_in_X = stimul_features.loc[X.index]
    y_answer = stimuli_features_in_X['answer']

    # consistent participant split for CB (shared by all CB tasks)
    ind_train = [i in train_participants for i in participants_in_X]
    ind_test  = [i in test_participants for i in participants_in_X]

    X_train, X_test = X[ind_train], X[ind_test]
    stimuli_train, stimuli_test = stimuli_features_in_X[ind_train], stimuli_features_in_X[ind_test]

    # ---------- SAVE feature frames ONCE per CB split (no {name} in filenames)
    X_train.to_csv(f'{base}/X_train.csv')
    X_test.to_csv(f'{base}/X_test.csv')

    stimuli_train.to_csv(f'{base}/stimul_features_train.csv')
    stimuli_test.to_csv(f'{base}/stimul_features_test.csv')

    # length-normalised
    X_len = X.copy()
    for c in X_len.columns:
        if ('sum' in c) or ('count' in c):
            X_len[c] = X_len[c] / stimuli_features_in_X['text_length']
    X_len_train, X_len_test = X_len[ind_train], X_len[ind_test]
    X_len_train.to_csv(f'{base}/X_length_normalised_train.csv')
    X_len_test.to_csv(f'{base}/X_length_normalised_test.csv')

    # participant-normalised (per participant stats)
    df_tr = X_train.copy()
    df_tr['participant'] = [i.split('_')[0] for i in df_tr.index]
    df_tr = df_tr.set_index('participant')
    p_mean_tr = df_tr.groupby('participant').mean()
    p_std_tr  = df_tr.groupby('participant').std()

    X_part_tr = (df_tr - p_mean_tr) / p_std_tr
    X_part_tr.index = X_train.index

    df_te = X_test.copy()
    df_te['participant'] = [i.split('_')[0] for i in df_te.index]
    df_te = df_te.set_index('participant')
    p_mean_te = df_te.groupby('participant').mean()
    p_std_te  = df_te.groupby('participant').std()
    X_part_te = (df_te - p_mean_te) / p_std_te
    X_part_te.index = X_test.index

    X_part_tr.to_csv(f'{base}/X_participants_normalised_train.csv')
    X_part_te.to_csv(f'{base}/X_participants_normalised_test.csv')

    # length -> participant normalised
    df_xl_tr = X_len_train.copy()
    df_xl_tr['participant'] = [i.split('_')[0] for i in df_xl_tr.index]
    df_xl_tr = df_xl_tr.set_index('participant')
    lp_mean_tr = df_xl_tr.groupby('participant').mean()
    lp_std_tr  = df_xl_tr.groupby('participant').std()
    X_len_part_tr = (df_xl_tr - lp_mean_tr) / lp_std_tr

    df_xl_te = X_len_test.copy()
    df_xl_te['participant'] = [i.split('_')[0] for i in df_xl_te.index]
    df_xl_te = df_xl_te.set_index('participant')
    lp_mean_te = df_xl_te.groupby('participant').mean()
    lp_std_te  = df_xl_te.groupby('participant').std()
    X_len_part_te = (df_xl_te - lp_mean_te) / lp_std_te

    X_len_part_tr.index = X_train.index
    X_len_part_te.index = X_test.index
    X_len_part_tr.to_csv(f'{base}/X_length_participants_normalised_train.csv')
    X_len_part_te.to_csv(f'{base}/X_length_participants_normalised_test.csv')

    # TEST normalised by "mean-of-means / mean-of-stds" from TRAIN participants
    eps = 1e-12
    g_mean_base = p_mean_tr.mean(axis=0)
    g_std_base  = p_std_tr.fillna(0.0).mean(axis=0).replace(0.0, np.nan).fillna(eps)
    ( (X_test - g_mean_base) / g_std_base ).to_csv(
        f'{base}/X_participants_normalised_test_bytrainavg.csv'
    )

    g_mean_len = lp_mean_tr.mean(axis=0)
    g_std_len  = lp_std_tr.fillna(0.0).mean(axis=0).replace(0.0, np.nan).fillna(eps)
    ( (X_len_test - g_mean_len) / g_std_len ).to_csv(
        f'{base}/X_length_participants_normalised_test_bytrainavg.csv'
    )

    # stimulus-normalised (by train participants; with global fallback)
    by_stim_mean, by_stim_std, overall_mean, overall_std = _build_stimulus_norm(X, train_participants)
    X_stim_tr = _apply_stimulus_norm(X_train, by_stim_mean, by_stim_std, overall_mean, overall_std)
    X_stim_te = _apply_stimulus_norm(X_test,  by_stim_mean, by_stim_std, overall_mean, overall_std)
    X_stim_tr.to_csv(f'{base}/X_stimulus_normalised_train.csv')
    X_stim_te.to_csv(f'{base}/X_stimulus_normalised_test.csv')

    X_len_stim_tr = _apply_stimulus_norm(X_len_train, by_stim_mean, by_stim_std, overall_mean, overall_std)
    X_len_stim_te = _apply_stimulus_norm(X_len_test,  by_stim_mean, by_stim_std, overall_mean, overall_std)
    X_len_stim_tr.to_csv(f'{base}/X_length_stimulus_normalised_train.csv')
    X_len_stim_te.to_csv(f'{base}/X_length_stimulus_normalised_test.csv')

    # ---------- SAVE targets PER TASK (use {name})
    if name == 'binary':
        y_binary[ind_train].to_csv(f'{base}/y_binary_train.csv')
        y_binary[ind_test].to_csv(f'{base}/y_binary_test.csv')
    elif name == 'multiclass':
        y_multiclass[ind_train].to_csv(f'{base}/y_multiclass_train.csv')
        y_multiclass[ind_test].to_csv(f'{base}/y_multiclass_test.csv')

    # always save regression/answer once (shared)
    y_reg[ind_train].to_csv(f'{base}/y_reg_train.csv')
    y_reg[ind_test].to_csv(f'{base}/y_reg_test.csv')
    y_answer[ind_train].to_csv(f'{base}/y_answer_train.csv')
    y_answer[ind_test].to_csv(f'{base}/y_answer_test.csv')

In [26]:
split_cb_and_save(X_screen, 'screen', name='binary')
split_cb_and_save(X_screen, 'screen', name='multiclass')

split_cb_and_save(X_answer, 'answer', name='binary')
split_cb_and_save(X_answer, 'answer', name='multiclass')

### Альтернативное определение таргета (match_mismatch) и разбиение данных

In [19]:
stimul_features['TextID'] = [int(i[1]) for i in stimul_features.index.str.split('_')]

train_stimuli, test_stimuli = train_test_split(
    stimul_features.drop_duplicates(subset=['TextID'])['TextID'].tolist(),
    test_size=0.3,
    stratify=stimul_features.drop_duplicates(subset=['TextID'])['valence'].tolist(),
    random_state=RANDOM_STATE
    )

stimul_features = stimul_features.drop(columns=['TextID'])

In [28]:
def split_mm_and_save(X, X_name, name):
    # name is 'binary' or 'multiclass' (used only for targets)
    base = f'datasets_splitted_{X_name}_match_mismatch'
    os.makedirs(base, exist_ok=True)

    participants_in_X = [int(i.split('_')[0]) for i in X.index]
    stimuls_in_X      = [int(i.split('_')[1]) for i in X.index]

    y_reg = pd.Series([participants_meta[participants_meta['ID']==i]['IAT_results'].item()
                       for i in participants_in_X], name='IAT_score', index=X.index)
    y_binary = pd.Series([participants_meta[participants_meta['ID']==i]['IAT_result_word'].item()
                          for i in participants_in_X], name='IAT_binary_label', index=X.index).map({'Negative':0,'Positive':1})
    y_multiclass = pd.Series([participants_meta[participants_meta['ID']==i]['IAT_results2'].item()
                              for i in participants_in_X], name='IAT_multi_label', index=X.index).map({'Negative':0,'Positive':1,'Neutral':2})

    stimuli_features_in_X = stimul_features.loc[X.index]
    y_answer = stimuli_features_in_X['answer']

    # build mm labels
    y_multiclass_mm = []
    for ib, val in zip(y_binary, stimuli_features_in_X['valence']):
        y_multiclass_mm.append(2 if val==2 else (1 if ib==val else 0))
    y_multiclass_mm = pd.Series(y_multiclass_mm, index=X.index)

    y_binary_mm = []
    for imc, val in zip(y_multiclass, stimuli_features_in_X['valence']):
        y_binary_mm.append(1 if imc==val else 0)
    y_binary_mm = pd.Series(y_binary_mm, index=X.index)

    # split for MM (participants x stimuli)
    ind_train_mm = [(i in train_participants) and (j in train_stimuli) for i, j in zip(participants_in_X, stimuls_in_X)]
    ind_test_mm  = [(i in test_participants)  and (j in test_stimuli)  for i, j in zip(participants_in_X, stimuls_in_X)]

    X_tr, X_te = X[ind_train_mm], X[ind_test_mm]
    stimuli_tr, stimuli_te = stimuli_features_in_X[ind_train_mm], stimuli_features_in_X[ind_test_mm]

    # ---------- SAVE feature frames ONCE per MM split (no {name})
    X_tr.to_csv(f'{base}/X_train.csv')
    X_te.to_csv(f'{base}/X_test.csv')
    stimuli_tr.to_csv(f'{base}/stimul_features_train.csv')
    stimuli_te.to_csv(f'{base}/stimul_features_test.csv')

    # length-normalised
    X_len = X.copy()
    for c in X.columns:
        if ('sum' in c) or ('count' in c):
            X_len[c] = X_len[c] / stimuli_features_in_X['text_length']
    X_len_tr, X_len_te = X_len[ind_train_mm], X_len[ind_test_mm]
    X_len_tr.to_csv(f'{base}/X_length_normalised_train.csv')
    X_len_te.to_csv(f'{base}/X_length_normalised_test.csv')

    # participant-normalised
    df_tr = X_tr.copy()
    df_tr['participant'] = [i.split('_')[0] for i in df_tr.index]
    df_tr = df_tr.set_index('participant')
    p_mean_tr = df_tr.groupby('participant').mean()
    p_std_tr  = df_tr.groupby('participant').std()
    X_part_tr = (df_tr - p_mean_tr) / p_std_tr

    df_te = X_te.copy()
    df_te['participant'] = [i.split('_')[0] for i in df_te.index]
    df_te = df_te.set_index('participant')
    p_mean_te = df_te.groupby('participant').mean()
    p_std_te  = df_te.groupby('participant').std()
    X_part_te = (df_te - p_mean_te) / p_std_te

    X_part_tr.index = X_tr.index
    X_part_te.index = X_te.index
    X_part_tr.to_csv(f'{base}/X_participants_normalised_train.csv')
    X_part_te.to_csv(f'{base}/X_participants_normalised_test.csv')

    # length -> participant normalised
    df_xl_tr = X_len_tr.copy()
    df_xl_tr['participant'] = [i.split('_')[0] for i in df_xl_tr.index]
    df_xl_tr = df_xl_tr.set_index('participant')
    lp_mean_tr = df_xl_tr.groupby('participant').mean()
    lp_std_tr  = df_xl_tr.groupby('participant').std()
    X_len_part_tr = (df_xl_tr - lp_mean_tr) / lp_std_tr

    df_xl_te = X_len_te.copy()
    df_xl_te['participant'] = [i.split('_')[0] for i in df_xl_te.index]
    df_xl_te = df_xl_te.set_index('participant')
    lp_mean_te = df_xl_te.groupby('participant').mean()
    lp_std_te  = df_xl_te.groupby('participant').std()
    X_len_part_te = (df_xl_te - lp_mean_te) / lp_std_te

    X_len_part_tr.index = X_tr.index
    X_len_part_te.index = X_te.index
    X_len_part_tr.to_csv(f'{base}/X_length_participants_normalised_train.csv')
    X_len_part_te.to_csv(f'{base}/X_length_participants_normalised_test.csv')

    # ---------- SAVE targets PER TASK
    if name == 'binary':
        y_binary_mm[ind_train_mm].to_csv(f'{base}/y_binary_train.csv')
        y_binary_mm[ind_test_mm].to_csv(f'{base}/y_binary_test.csv')
    elif name == 'multiclass':
        y_multiclass_mm[ind_train_mm].to_csv(f'{base}/y_multiclass_train.csv')
        y_multiclass_mm[ind_test_mm].to_csv(f'{base}/y_multiclass_test.csv')

    # always save regression/answer for MM too (if you need them downstream)
    y_reg[ind_train_mm].to_csv(f'{base}/y_reg_train.csv')
    y_reg[ind_test_mm].to_csv(f'{base}/y_reg_test.csv')
    y_answer[ind_train_mm].to_csv(f'{base}/y_answer_train.csv')
    y_answer[ind_test_mm].to_csv(f'{base}/y_answer_test.csv')

In [29]:
split_mm_and_save(X_screen, 'screen', name='multiclass') #наоборот - так и должно быть
split_mm_and_save(X_screen, 'screen', name='binary') #наоборот - так и должно быть

split_mm_and_save(X_answer, 'answer', name='multiclass') #наоборот - так и должно быть
split_mm_and_save(X_answer, 'answer', name='binary') #наоборот - так и должно быть

# EXP

In [20]:
participants_meta['exp_general_binary'] = (participants_meta['EXP_general']<3.5).astype(int)
participants_meta['exp_general_multi'] = 2
participants_meta['exp_general_multi'][(participants_meta['EXP_general']<2.5)]=1
participants_meta['exp_general_multi'][(participants_meta['EXP_general']>4.5)]=0

C:\Users\LEGION\AppData\Local\Temp\ipykernel_32936\4258137436.py:3: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  participants_meta['exp_general_multi'][(participants_meta['EXP_general']<2.5)]=1
C:\Users\LEGION\AppData\Local\Temp\ipykernel_3

In [21]:
def split_mm_exp_and_save(X, X_name, name):
    # name is 'binary' or 'multiclass' (used only for targets)
    base = f'datasets_splitted_{X_name}_match_mismatch_general'
    os.makedirs(base, exist_ok=True)

    participants_in_X = [int(i.split('_')[0]) for i in X.index]
    stimuls_in_X      = [int(i.split('_')[1]) for i in X.index]

    y_reg = pd.Series([participants_meta[participants_meta['ID']==i]['IAT_results'].item()
                       for i in participants_in_X], name='IAT_score', index=X.index)
    y_binary = pd.Series([participants_meta[participants_meta['ID']==i]['exp_general_binary'].item()
                          for i in participants_in_X], name='IAT_binary_label', index=X.index)
    y_multiclass = pd.Series([participants_meta[participants_meta['ID']==i]['exp_general_multi'].item()
                              for i in participants_in_X], name='IAT_multi_label', index=X.index)

    stimuli_features_in_X = stimul_features.loc[X.index]
    y_answer = stimuli_features_in_X['answer']

    # build mm labels
    y_multiclass_mm = []
    for ib, val in zip(y_binary, stimuli_features_in_X['valence']):
        y_multiclass_mm.append(2 if val==2 else (1 if ib==val else 0))
    y_multiclass_mm = pd.Series(y_multiclass_mm, index=X.index)

    y_binary_mm = []
    for imc, val in zip(y_multiclass, stimuli_features_in_X['valence']):
        y_binary_mm.append(1 if imc==val else 0)
    y_binary_mm = pd.Series(y_binary_mm, index=X.index)

    # split for MM (participants x stimuli)
    ind_train_mm = [(i in train_participants) and (j in train_stimuli) for i, j in zip(participants_in_X, stimuls_in_X)]
    ind_test_mm  = [(i in test_participants)  and (j in test_stimuli)  for i, j in zip(participants_in_X, stimuls_in_X)]

    X_tr, X_te = X[ind_train_mm], X[ind_test_mm]
    stimuli_tr, stimuli_te = stimuli_features_in_X[ind_train_mm], stimuli_features_in_X[ind_test_mm]

    # ---------- SAVE feature frames ONCE per MM split (no {name})
    X_tr.to_csv(f'{base}/X_train.csv')
    X_te.to_csv(f'{base}/X_test.csv')
    stimuli_tr.to_csv(f'{base}/stimul_features_train.csv')
    stimuli_te.to_csv(f'{base}/stimul_features_test.csv')

    # length-normalised
    X_len = X.copy()
    for c in X.columns:
        if ('sum' in c) or ('count' in c):
            X_len[c] = X_len[c] / stimuli_features_in_X['text_length']
    X_len_tr, X_len_te = X_len[ind_train_mm], X_len[ind_test_mm]
    X_len_tr.to_csv(f'{base}/X_length_normalised_train.csv')
    X_len_te.to_csv(f'{base}/X_length_normalised_test.csv')

    # participant-normalised
    df_tr = X_tr.copy()
    df_tr['participant'] = [i.split('_')[0] for i in df_tr.index]
    df_tr = df_tr.set_index('participant')
    p_mean_tr = df_tr.groupby('participant').mean()
    p_std_tr  = df_tr.groupby('participant').std()
    X_part_tr = (df_tr - p_mean_tr) / p_std_tr

    df_te = X_te.copy()
    df_te['participant'] = [i.split('_')[0] for i in df_te.index]
    df_te = df_te.set_index('participant')
    p_mean_te = df_te.groupby('participant').mean()
    p_std_te  = df_te.groupby('participant').std()
    X_part_te = (df_te - p_mean_te) / p_std_te

    X_part_tr.index = X_tr.index
    X_part_te.index = X_te.index
    X_part_tr.to_csv(f'{base}/X_participants_normalised_train.csv')
    X_part_te.to_csv(f'{base}/X_participants_normalised_test.csv')

    # length -> participant normalised
    df_xl_tr = X_len_tr.copy()
    df_xl_tr['participant'] = [i.split('_')[0] for i in df_xl_tr.index]
    df_xl_tr = df_xl_tr.set_index('participant')
    lp_mean_tr = df_xl_tr.groupby('participant').mean()
    lp_std_tr  = df_xl_tr.groupby('participant').std()
    X_len_part_tr = (df_xl_tr - lp_mean_tr) / lp_std_tr

    df_xl_te = X_len_te.copy()
    df_xl_te['participant'] = [i.split('_')[0] for i in df_xl_te.index]
    df_xl_te = df_xl_te.set_index('participant')
    lp_mean_te = df_xl_te.groupby('participant').mean()
    lp_std_te  = df_xl_te.groupby('participant').std()
    X_len_part_te = (df_xl_te - lp_mean_te) / lp_std_te

    X_len_part_tr.index = X_tr.index
    X_len_part_te.index = X_te.index
    X_len_part_tr.to_csv(f'{base}/X_length_participants_normalised_train.csv')
    X_len_part_te.to_csv(f'{base}/X_length_participants_normalised_test.csv')

    # ---------- SAVE targets PER TASK
    if name == 'binary':
        y_binary_mm[ind_train_mm].to_csv(f'{base}/y_binary_train.csv')
        y_binary_mm[ind_test_mm].to_csv(f'{base}/y_binary_test.csv')
    elif name == 'multiclass':
        y_multiclass_mm[ind_train_mm].to_csv(f'{base}/y_multiclass_train.csv')
        y_multiclass_mm[ind_test_mm].to_csv(f'{base}/y_multiclass_test.csv')

    # always save regression/answer for MM too (if you need them downstream)
    y_reg[ind_train_mm].to_csv(f'{base}/y_reg_train.csv')
    y_reg[ind_test_mm].to_csv(f'{base}/y_reg_test.csv')
    y_answer[ind_train_mm].to_csv(f'{base}/y_answer_train.csv')
    y_answer[ind_test_mm].to_csv(f'{base}/y_answer_test.csv')

In [22]:
split_mm_exp_and_save(X_screen, 'screen', name='multiclass') #наоборот - так и должно быть
split_mm_exp_and_save(X_screen, 'screen', name='binary') #наоборот - так и должно быть

split_mm_exp_and_save(X_answer, 'answer', name='multiclass') #наоборот - так и должно быть
split_mm_exp_and_save(X_answer, 'answer', name='binary') #наоборот - так и должно быть